# Explore a regulation's structured JSON
Run **Kernel → Restart & Run All**. This viewer reads local extraction output; it does not call an LLM, load `.env`, or change your source graph.

The structure graph contains source elements and proposed `CONTAINS` relationships, not verified applicability decisions. Page references and extracted clause labels are shown as recorded. Install dependencies once in the kernel if needed.

In [ ]:
%pip install pandas ipywidgets

In [ ]:
from pathlib import Path
import json
from collections import Counter, defaultdict
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, JSON, clear_output
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "pdf-audit-and-extract").is_dir() and (p / ".git").exists()), None)
if PROJECT is None:
    raise RuntimeError("Start Jupyter inside the certivity-hack repository.")
# Change this path to inspect another extracted PDF/version.
JSON_PATH = PROJECT / "pipeline_output/json-structure/UN_R150/base/UN_R150_base/document.structure.json"
if not JSON_PATH.exists():
    raise FileNotFoundError(f"Run the extraction notebook first, or update JSON_PATH: {JSON_PATH}")
graph = json.loads(JSON_PATH.read_text())
assert isinstance(graph.get("nodes"), list) and isinstance(graph.get("edges"), list)
nodes = {n['id']: n for n in graph['nodes']}
assert len(nodes) == len(graph['nodes']), "Duplicate node IDs found"
children, parents = defaultdict(list), defaultdict(list)
for edge in graph['edges']:
    if edge['relation'] == 'CONTAINS':
        children[edge['from']].append(edge['to'])
        parents[edge['to']].append(edge['from'])
def summary(node):
    return {'id':node['id'], 'type':node['type'], 'context':node.get('context',''),
            'clause':node.get('clause_label',''), 'pages':node.get('source',{}).get('pages',[]),
            'text':node.get('text','')}
frame=pd.DataFrame([summary(n) for n in nodes.values()])
print(f"Loaded {JSON_PATH.name}: {len(nodes):,} nodes and {len(graph['edges']):,} edges")
display(JSON(graph.get('document',{}), expanded=False))
display(frame.groupby('type').size().rename('count').to_frame())


## Search and inspect
Filter by text, node ID, clause label, context, type, or PDF page. Select a result to view its full text, source provenance, raw JSON, and immediate relatives. A blank clause field can mean the parser did not recover numbering. It does not mean the source lacks a clause number.

In [ ]:
query=widgets.Text(description='Search:',placeholder='e.g. water, 4.1.1, Annex 5')
type_filter=widgets.Dropdown(description='Type:',options=['All']+sorted(frame['type'].unique().tolist()))
page_filter=widgets.IntText(description='Page:',value=0)
selector=widgets.Select(description='Node:',options=[],rows=12,layout=widgets.Layout(width='100%'))
status=widgets.HTML()
detail=widgets.Output()
def relatives_table(ids):
    return pd.DataFrame([summary(nodes[i]) for i in ids if i in nodes])
def show_node(change=None):
    with detail:
        clear_output(wait=True)
        if selector.value is None: return
        n=nodes[selector.value]
        print(n['id'], '|', n['type'], '| pages:', n.get('source',{}).get('pages',[]))
        print(n.get('text',''))
        display(JSON(n,expanded=False))
        for title,ids in [('Parents',parents[n['id']]),('Children',children[n['id']])]:
            print(title)
            if ids: display(relatives_table(ids))
            else: print('(none recorded)')
        if 'table' in n:
            print('Table cells (including row/column offsets where available)')
            display(pd.DataFrame(n['table'].get('table_cells',[])))
def refresh(change=None):
    q=query.value.casefold().strip()
    matches=[n for n in nodes.values()
        if (type_filter.value=='All' or n['type']==type_filter.value)
        and (page_filter.value<=0 or page_filter.value in n.get('source',{}).get('pages',[]))
        and (not q or q in ' '.join([n['id'],n.get('text',''),n.get('context',''),n.get('clause_label','')]).casefold())]
    selector.options=[(f"{n['id']} | {n.get('clause_label','')} | {n.get('text','')[:95]}",n['id']) for n in matches[:500]]
    status.value=f"{len(matches)} matches; showing up to 500. Page 0 means all pages."
    if selector.options: selector.value=selector.options[0][1]
    show_node()
for control in [query,type_filter,page_filter]:control.observe(refresh,names='value')
selector.observe(show_node,names='value')
display(widgets.VBox([query,widgets.HBox([type_filter,page_filter]),status,selector,detail]))
refresh()


## Inspect a branch as text
Set `ROOT_NODE` to any ID from the explorer. This displays a bounded hierarchy with cycle detection. These edges describe document organisation, not logical applicability.

In [ ]:
ROOT_NODE = next((n['id'] for n in nodes.values() if n.get('clause_label')=='4.1.1' and n.get('context')=='main'), next(iter(nodes)))
def print_branch(node_id, max_depth=3, max_nodes=50):
    seen=set(); count=0
    def walk(current,depth):
        nonlocal count
        if count>=max_nodes: return
        count+=1
        n=nodes.get(current)
        if n is None: print('  '*depth+f'{current} [missing]'); return
        print('  '*depth+f"{current} [{n.get('clause_label','')}] {n.get('text','')[:110]}")
        if current in seen: print('  '*depth+'[already visited]'); return
        seen.add(current)
        if depth<max_depth:
            for child in children[current]:walk(child,depth+1)
    walk(node_id,0)
print_branch(ROOT_NODE)


## Raw JSON and structural checks
Use this cell even if widgets do not render in your Jupyter frontend. Structural checks do not assess whether obligations or applicability conditions were correctly extracted.

In [ ]:
NODE_ID = ROOT_NODE
print(json.dumps(nodes[NODE_ID],indent=2,ensure_ascii=False))
missing_edges=[e for e in graph['edges'] if e['from'] not in nodes or e['to'] not in nodes]
checks={'nodes':len(nodes),'edges':len(graph['edges']),
        'nodes_without_pages':sum(not n.get('source',{}).get('pages') for n in nodes.values()),
        'nodes_with_clause_labels':sum(bool(n.get('clause_label')) for n in nodes.values()),
        'dangling_edges':len(missing_edges),'recorded_warnings':len(graph.get('warnings',[]))}
display(JSON(checks))
if missing_edges:display(JSON(missing_edges))
if graph.get('warnings'):display(JSON(graph['warnings']))
